<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/05-files-and-data/01_lesson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 5: Files and Data

Up to now, every dataset in this course has been generated inside the notebook. Real projects start with files: CSVs exported from a database, JSON from an API, logs, folders of images. And real files are messy. Values are missing, units are mixed, someone typed "adelie" in one row and "ADELIE" in the next, and a stray comma shifts half a row into the wrong columns.

This module covers how to read and write data files properly, and how to turn messy raw data into something a model can use. We'll work with a real dataset: measurements of 344 penguins from Antarctica, collected by scientists at Palmer Station.

Everything is still pure Python, using the standard library's `pathlib`, `csv` and `json` modules. Module 7 introduces pandas, which does much of this in fewer lines, but pandas makes the same decisions you'll learn to make here, and when it guesses wrong, you need to know what it did.

**Contents**

1. Paths with `pathlib`
2. Reading and writing text files
3. Encodings: when text isn't just text
4. Downloading and caching a dataset
5. CSV files
6. Exploring a real dataset
7. Cleaning messy data
8. JSON and JSON Lines
9. A word about pickle
10. Putting it together: a reproducible data pipeline

**A note on Colab:** files you create in Colab live in a temporary folder (`/content`) that's wiped when the session ends. That's fine for this module, since everything is recreated when you rerun the notebook. For your own projects, save anything you want to keep to Google Drive or download it.

## 1. Paths with pathlib

A path says where a file is. You could treat paths as plain strings and glue them together with `"/"`, but that breaks on Windows (which uses `\`) and gets messy quickly. `pathlib.Path` handles all of it:

In [1]:
from pathlib import Path

work = Path("work")
data_file = work / "raw" / "penguins.csv"     # / joins path parts

print(data_file)
print(data_file.name)       # the file name
print(data_file.stem)       # name without the extension
print(data_file.suffix)     # the extension
print(data_file.parent)     # the folder it's in
print(data_file.parts)

work/raw/penguins.csv
penguins.csv
penguins
.csv
work/raw
('work', 'raw', 'penguins.csv')


Using `/` to join paths is an operator-overloading trick, exactly like the ones from Module 3: `Path` defines `__truediv__`.

None of that touched the disk. These are just descriptions of locations. To actually check or create things:

In [2]:
print(Path.cwd())                  # the current working directory
print(data_file.exists())

(work / "raw").mkdir(parents=True, exist_ok=True)   # create folders, no error if they exist
print((work / "raw").is_dir())

/content
False
True


`parents=True` creates any missing folders along the way, and `exist_ok=True` means "don't complain if it's already there". You'll write `mkdir(parents=True, exist_ok=True)` constantly.

**Relative vs absolute paths.** `Path("work")` is *relative*: it means "a folder called `work` inside the current working directory". In Colab that's `/content`. `.resolve()` turns it into an *absolute* path, the full location from the root of the disk:

In [3]:
print(work.resolve())

/content/work


Relative paths are what you want in project code, so it runs on anyone's machine. Hard-coding something like `C:/Users/anup/Desktop/data.csv` works only on one computer.

## 2. Reading and writing text files

The basic tool is `open()`, always used with `with`, so the file is closed even if something goes wrong (the context managers from Module 4):

In [4]:
notes = work / "notes.txt"

with open(notes, "w", encoding="utf-8") as f:
    f.write("run 1: accuracy 0.82\n")
    f.write("run 2: accuracy 0.85\n")

with open(notes, "a", encoding="utf-8") as f:     # "a" appends instead of overwriting
    f.write("run 3: accuracy 0.88\n")

with open(notes, encoding="utf-8") as f:           # "r" (read) is the default
    print(f.read())

run 1: accuracy 0.82
run 2: accuracy 0.85
run 3: accuracy 0.88



The modes you'll use:

| mode | meaning |
|---|---|
| `"r"` | read (default). Error if the file doesn't exist. |
| `"w"` | write. **Creates the file, or wipes it if it already exists.** |
| `"a"` | append to the end. Creates the file if needed. |
| `"x"` | create a new file. Error if it already exists. |
| add `"b"` | binary, for non-text files like images: `"rb"`, `"wb"` |

That warning about `"w"` deserves emphasis: opening an existing file in `"w"` mode empties it immediately, before you've written anything. If you're saving results you can't regenerate, `"x"` mode is a cheap safety net.

### Reading line by line

`f.read()` loads the whole file into one string. For big files, loop over the file object instead. A file object is an iterator that yields one line at a time, so memory use stays flat no matter how big the file is, just like the generators in Module 4:

In [5]:
with open(notes, encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        print(line_number, repr(line))

1 'run 1: accuracy 0.82\n'
2 'run 2: accuracy 0.85\n'
3 'run 3: accuracy 0.88\n'


Each line still has its `\n` on the end, which is why you'll almost always see `line.strip()` or `line.rstrip("\n")` right after.

For small files, `pathlib` has shortcuts that open, read or write, and close in one call:

In [6]:
text = notes.read_text(encoding="utf-8")
print(text.splitlines())

(work / "hello.txt").write_text("hello\n", encoding="utf-8")
print((work / "hello.txt").read_text(encoding="utf-8"))

['run 1: accuracy 0.82', 'run 2: accuracy 0.85', 'run 3: accuracy 0.88']
hello



### Always pass encoding="utf-8"

You'll notice `encoding="utf-8"` on every call above. If you leave it out, Python uses your operating system's default, which is UTF-8 on Linux and Mac (and in Colab) but often something else on Windows. Code that works on your laptop then breaks on a colleague's. Always being explicit costs nothing. The next section shows what goes wrong when you aren't.

## 3. Encodings: when text isn't just text

Files store **bytes**, numbers from 0 to 255. Text is made of **characters**. An *encoding* is the rule for turning one into the other. For plain English letters, nearly every encoding agrees, which is why problems often hide until someone writes a name with an accent:

In [7]:
name = "Adélie"
print(name.encode("utf-8"))      # é takes two bytes in UTF-8
print(name.encode("latin-1"))    # and one byte in Latin-1

b'Ad\xc3\xa9lie'
b'Ad\xe9lie'


Same text, different bytes. Now suppose a colleague exports a file with an older tool that uses Latin-1, and you read it as UTF-8:

In [8]:
old_file = work / "old_export.csv"
old_file.write_text("species\nAdélie\n", encoding="latin-1")

try:
    old_file.read_text(encoding="utf-8")
except UnicodeDecodeError as e:
    print("UnicodeDecodeError:", e)

print(old_file.read_text(encoding="latin-1"))

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe9 in position 10: invalid continuation byte
species
Adélie



The fix is to read the file with the encoding it was written in. If you don't know what that is, UTF-8 is the best first guess these days. If that fails, `latin-1` (also called ISO-8859-1) and `cp1252` (the old Windows default) are the usual suspects.

You'll see one more encoding problem in practice. Excel often saves "UTF-8" CSV files with an invisible marker at the very start, called a *byte order mark* (BOM). Read with plain `"utf-8"`, it sticks to the first column name:

In [9]:
excel_file = work / "from_excel.csv"
excel_file.write_text("id,species\n1,Gentoo\n", encoding="utf-8-sig")   # adds a BOM

header = excel_file.read_text(encoding="utf-8").splitlines()[0]
print(repr(header))

header = excel_file.read_text(encoding="utf-8-sig").splitlines()[0]
print(repr(header))

'\ufeffid,species'
'id,species'


With plain `utf-8`, the first column is called `'\ufeffid'`, not `'id'`, so `row["id"]` raises a `KeyError` even though the file plainly has an `id` column. Reading with `"utf-8-sig"` removes the BOM if it's there and does nothing if it isn't, so it's a safe choice for CSV files that might have come from Excel.

## 4. Downloading and caching a dataset

Our dataset is the Palmer Penguins data, a popular modern alternative to the classic Iris dataset. It records the species, island, bill size, flipper length, body mass and sex of 344 penguins observed between 2007 and 2009. It was collected by Dr. Kristen Gorman and the Palmer Station Long Term Ecological Research program, and packaged by Allison Horst, Alison Hill and Kristen Gorman. It's released under CC0, so it can be shared freely, and a copy lives in this repository's `data` folder.

A good data-loading function follows one simple rule: **download once, then reuse the local copy**. Downloads are slow, servers go down, and you don't want to fetch the same file every time you rerun a notebook.

In [10]:
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/anupbagale/python-for-ai-ml/main/05-files-and-data/data/penguins.csv"
DATA_DIR = Path("data")


def get_penguins() -> Path:
    """Return the path to penguins.csv, downloading it only if it isn't already here."""
    path = DATA_DIR / "penguins.csv"
    if not path.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print(f"downloading {DATA_URL}")
        urllib.request.urlretrieve(DATA_URL, path)
    return path


penguins_path = get_penguins()
print(penguins_path, penguins_path.stat().st_size, "bytes")

data/penguins.csv 15241 bytes


In Colab, the first run downloads the file and later runs reuse it. If you cloned the repository and opened the notebook locally, the file is already there and nothing is downloaded.

This is the same pattern that `torchvision.datasets`, Hugging Face `datasets` and scikit-learn's `fetch_*` functions use, with a cache folder on disk. A robust version would also check that the download is complete, for example by comparing a checksum, since a dropped connection can leave you with half a file.

## 5. CSV files

CSV (comma-separated values) is the most common format for tabular data. Let's look at the first few lines of the penguin file:

In [11]:
with open(penguins_path, encoding="utf-8") as f:
    for _ in range(5):
        print(f.readline().rstrip())

species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
Adelie,Torgersen,39.1,18.7,181,3750,male,2007
Adelie,Torgersen,39.5,17.4,186,3800,female,2007
Adelie,Torgersen,40.3,18,195,3250,female,2007
Adelie,Torgersen,NA,NA,NA,NA,NA,2007


It looks simple enough to parse with `line.split(",")`. Don't. CSV has rules for values that contain commas, quotes or line breaks, and splitting by hand breaks on the first one:

In [12]:
tricky = 'name,city,score\n"Bagale, Anup",Kathmandu,0.91\n"O\'Brien ""OB""",Dublin,0.85\n'
(work / "tricky.csv").write_text(tricky, encoding="utf-8")

print("split by hand:")
for line in tricky.splitlines():
    print(" ", line.split(","))

split by hand:
  ['name', 'city', 'score']
  ['"Bagale', ' Anup"', 'Kathmandu', '0.91']
  ['"O\'Brien ""OB"""', 'Dublin', '0.85']


The quoted name `"Bagale, Anup"` got split in two, so that row has four fields instead of three, and every value after it is in the wrong column. The `csv` module follows the actual rules:

In [13]:
import csv

print("csv module:")
with open(work / "tricky.csv", encoding="utf-8", newline="") as f:
    for row in csv.reader(f):
        print(" ", row)

csv module:
  ['name', 'city', 'score']
  ['Bagale, Anup', 'Kathmandu', '0.91']
  ['O\'Brien "OB"', 'Dublin', '0.85']


Correct, including the doubled quotes, which are CSV's way of writing a literal `"` inside a quoted value.

The `newline=""` argument is part of the `csv` module's instructions. It lets the `csv` module handle line endings itself, which matters for quoted values that contain line breaks, and it prevents a bug on Windows where every row gets written with a blank line after it.

### DictReader: rows as dictionaries

`csv.DictReader` uses the header row as keys, so each row becomes a dict. That's much easier to work with than remembering that column 4 is flipper length:

In [14]:
with open(penguins_path, encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    print(reader.fieldnames)
    first = next(reader)

print(first)

['species', 'island', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex', 'year']
{'species': 'Adelie', 'island': 'Torgersen', 'bill_length_mm': '39.1', 'bill_depth_mm': '18.7', 'flipper_length_mm': '181', 'body_mass_g': '3750', 'sex': 'male', 'year': '2007'}


### Everything is a string

Look closely at that row: `'bill_length_mm': '39.1'`. CSV has no types. Every value comes back as a string, including numbers and the `'NA'` that marks missing values. Converting is your job:

In [15]:
print(first["bill_length_mm"] + first["bill_depth_mm"])     # string concatenation, not addition

39.118.7


That's a quiet bug: no error, just a nonsense value. So we need a converter. Missing values in this file are written `NA`, but other files use an empty string, `?`, `-`, `null` or `N/A`, so we'll handle all of the common ones:

In [16]:
MISSING = {"", "na", "n/a", "nan", "null", "none", "?", "-"}


def to_number(text: str) -> float | None:
    """Convert a CSV field to a float, or None if it's a missing-value marker."""
    text = text.strip()
    if text.lower() in MISSING:
        return None
    return float(text)


NUMERIC = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]


def load_penguins(path: Path) -> list[dict]:
    rows = []
    with open(path, encoding="utf-8", newline="") as f:
        for raw in csv.DictReader(f):
            row = {
                "species": raw["species"],
                "island": raw["island"],
                "sex": None if raw["sex"].strip().lower() in MISSING else raw["sex"],
                "year": int(raw["year"]),
            }
            for col in NUMERIC:
                row[col] = to_number(raw[col])
            rows.append(row)
    return rows


penguins = load_penguins(penguins_path)
print(len(penguins), "penguins")
print(penguins[0])
print(penguins[3])

344 penguins
{'species': 'Adelie', 'island': 'Torgersen', 'sex': 'male', 'year': 2007, 'bill_length_mm': 39.1, 'bill_depth_mm': 18.7, 'flipper_length_mm': 181.0, 'body_mass_g': 3750.0}
{'species': 'Adelie', 'island': 'Torgersen', 'sex': None, 'year': 2007, 'bill_length_mm': None, 'bill_depth_mm': None, 'flipper_length_mm': None, 'body_mass_g': None}


Row 3 shows a penguin with no measurements at all. We'll decide what to do about those shortly.

Note that `to_number` doesn't catch errors. If a numeric column contains something unexpected like `"thirty"`, `float()` raises a `ValueError` and loading stops. That's deliberate: an unexpected value means something about the file isn't what we assumed, and we want to hear about it rather than silently turn it into a missing value.

### Writing CSV

`csv.DictWriter` is the mirror image of `DictReader`:

In [17]:
results = [
    {"model": "knn", "k": 5, "accuracy": 0.86},
    {"model": "logreg", "k": None, "accuracy": 0.88},
]

with open(work / "results.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["model", "k", "accuracy"])
    writer.writeheader()
    writer.writerows(results)

print((work / "results.csv").read_text(encoding="utf-8"))

model,k,accuracy
knn,5,0.86
logreg,,0.88



`None` is written as an empty field. Since everything becomes text, whoever reads the file has to convert types again. CSV doesn't remember that `accuracy` was a float.

## 6. Exploring a real dataset

Before cleaning or modelling anything, look at the data. Three questions come first in almost every project: how many rows, how are the classes distributed, and what's missing?

In [18]:
from collections import Counter

print("rows:", len(penguins))
print("species:", Counter(p["species"] for p in penguins))
print("islands:", Counter(p["island"] for p in penguins))
print("sex:", Counter(p["sex"] for p in penguins))

rows: 344
species: Counter({'Adelie': 152, 'Gentoo': 124, 'Chinstrap': 68})
islands: Counter({'Biscoe': 168, 'Dream': 124, 'Torgersen': 52})
sex: Counter({'male': 168, 'female': 165, None: 11})


The classes aren't balanced: there are more than twice as many Adelie penguins as Chinstraps. A model that always guessed "Adelie" would be right 44% of the time, so that's the baseline to beat. Eleven penguins have no recorded sex.

Now the missing values per column, and which rows have them:

In [19]:
columns = list(penguins[0])

for col in columns:
    n_missing = sum(p[col] is None for p in penguins)
    if n_missing:
        print(f"{col:18s} {n_missing:3d} missing")

no_measurements = [i for i, p in enumerate(penguins) if all(p[c] is None for c in NUMERIC)]
print("rows with no measurements at all:", no_measurements)

sex                 11 missing
bill_length_mm       2 missing
bill_depth_mm        2 missing
flipper_length_mm    2 missing
body_mass_g          2 missing
rows with no measurements at all: [3, 271]


Only two penguins are missing their measurements, and they're missing all four. Those rows carry no useful information for predicting species from measurements, so dropping them is the obvious choice. The missing `sex` values are a different case: the measurements are fine, so we'd lose good data by dropping those rows.

Some summary statistics per species, to see whether the measurements can tell the species apart:

In [20]:
def mean(values):
    values = [v for v in values if v is not None]
    return sum(values) / len(values)


print(f"{'species':10s}" + "".join(f"{c:>20s}" for c in NUMERIC))
for species in sorted({p["species"] for p in penguins}):
    group = [p for p in penguins if p["species"] == species]
    print(f"{species:10s}" + "".join(f"{mean(p[c] for p in group):20.1f}" for c in NUMERIC))

species         bill_length_mm       bill_depth_mm   flipper_length_mm         body_mass_g
Adelie                    38.8                18.3               190.0              3700.7
Chinstrap                 48.8                18.4               195.8              3733.1
Gentoo                    47.5                15.0               217.2              5076.0


Gentoos are clearly the heaviest, with the longest flippers and the shallowest bills. Adelies have much shorter bills than the other two. Chinstraps look like Adelies except for bill length. So bill length alone separates Adelie from the rest, and the other measurements separate Gentoo. That's a strong hint a classifier will do well, which the mini project will confirm.

Notice that `mean` skips `None` values. Most real-world statistics functions do something like this, and it's worth knowing whether the function you're using does. In pandas, `.mean()` skips missing values by default. In NumPy, `np.mean` returns `nan` if any value is missing.

## 7. Cleaning messy data

The penguin dataset is unusually tidy. Real raw data, especially anything typed in by people, looks more like this. Imagine field notes from several researchers, combined into one file:

In [21]:
messy_text = """penguin_id,species,island,body_mass,sex,date_seen
P001,Adelie,Torgersen,3750 g,male,2007-11-11
P002, adelie ,torgersen,3.8 kg,F,11/16/2007
P003,ADELIE,Torgersen,,female,2007-11-16
P004,Gentoo,Biscoe,5200g,M,2008-11-13
P005,Gentoo penguin,Biscoe,-1,male,2008-11-13
P006,Chinstrap,Dream,3500 g,?,2009-11-21
P007,Chinstrap,Dream,3,500 g,female,2009-11-21
P002, adelie ,torgersen,3.8 kg,F,11/16/2007
P008,Gentoo,Biscoe,NA,female,2008-11-14
P009,Adélie,Torgersen,3650 g,female,2007-11-16
"""
messy_path = work / "field_notes.csv"
messy_path.write_text(messy_text, encoding="utf-8")

483

Ten lines, and nearly every kind of problem you'll meet:

- **Inconsistent text:** `Adelie`, ` adelie `, `ADELIE`, `Adélie`, and `Gentoo penguin` instead of `Gentoo`
- **Mixed units and formats:** `3750 g`, `5200g`, `3.8 kg`
- **Different codes for the same thing:** `male`/`M`, `female`/`F`
- **Missing values, written three ways:** an empty field, `?` and `NA`
- **Impossible values:** a body mass of `-1`, probably a "don't know" code
- **A broken row:** P007's `3,500 g` has a comma that wasn't quoted, so the row has one field too many
- **A duplicate:** P002 appears twice
- **Two date formats:** `2007-11-11` and `11/16/2007`

The approach that works is: one small function per kind of problem, each returning either a clean value or `None`, plus a record of every issue found so you can review it afterwards. Silent fixes are dangerous. If 30% of a column turns out to be unparseable, you need to know.

### Text categories

Strip whitespace, lowercase, drop the extra word, and map every known spelling to one canonical name. A mapping dict is better than clever string rules: it's explicit, and anything not in it is flagged rather than guessed.

In [22]:
SPECIES = {"adelie": "Adelie", "adélie": "Adelie", "gentoo": "Gentoo", "chinstrap": "Chinstrap"}
SEX = {"male": "male", "m": "male", "female": "female", "f": "female"}


def clean_category(value: str, mapping: dict[str, str], suffix: str = "") -> str | None:
    key = value.strip().lower()
    if suffix and key.endswith(suffix):
        key = key[: -len(suffix)].strip()
    return mapping.get(key)          # None for anything unknown or missing


for raw in ["Adelie", " adelie ", "ADELIE", "Adélie", "Gentoo penguin", "Emperor"]:
    print(f"{raw!r:18} -> {clean_category(raw, SPECIES, suffix='penguin')!r}")

'Adelie'           -> 'Adelie'
' adelie '         -> 'Adelie'
'ADELIE'           -> 'Adelie'
'Adélie'           -> 'Adelie'
'Gentoo penguin'   -> 'Gentoo'
'Emperor'          -> None


"Emperor" isn't in the mapping, so it comes back as `None`. That's the right behaviour: it might be a typo, or a genuinely new species, and either way a person should look at it.

### Numbers with units

In [23]:
def parse_mass_grams(value: str) -> float | None:
    text = value.strip().lower().replace(" ", "")
    if text in MISSING:
        return None
    if text.endswith("kg"):
        grams = float(text[:-2]) * 1000
    elif text.endswith("g"):
        grams = float(text[:-1])
    else:
        grams = float(text)
    if not 1000 <= grams <= 10000:       # no real penguin in this study is outside this range
        return None
    return grams


for raw in ["3750 g", "5200g", "3.8 kg", "", "NA", "-1"]:
    print(f"{raw!r:10} -> {parse_mass_grams(raw)}")

'3750 g'   -> 3750.0
'5200g'    -> 5200.0
'3.8 kg'   -> 3800.0
''         -> None
'NA'       -> None
'-1'       -> None


The range check is **domain knowledge**, not something the data tells you. A sanity range like this catches placeholder values like `-1` or `9999`, typos like an extra zero, and unit mix-ups like grams entered as kilograms.

### Dates

The `datetime` module can parse a date in a given format. We try each known format in turn, and return a standard ISO date string (`YYYY-MM-DD`), which also sorts correctly as text:

In [24]:
from datetime import datetime

DATE_FORMATS = ["%Y-%m-%d", "%m/%d/%Y"]


def parse_date(value: str) -> str | None:
    for fmt in DATE_FORMATS:
        try:
            return datetime.strptime(value.strip(), fmt).date().isoformat()
        except ValueError:
            continue
    return None


print(parse_date("2007-11-11"), parse_date("11/16/2007"), parse_date("yesterday"))

2007-11-11 2007-11-16 None


One warning about dates: `11/06/2007` is the 6th of November in the US and the 11th of June in most other places. Nothing in the string tells you which. You have to know where the data came from, and you should never let a tool guess silently.

### The cleaning loop

Now we put it together. We read with `csv.reader` rather than `DictReader` here, because we need to check each row's field count before trusting which value is in which column:

In [25]:
def clean_field_notes(path: Path) -> tuple[list[dict], list[str]]:
    clean, issues, seen_ids = [], [], set()

    with open(path, encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        for line_no, fields in enumerate(reader, start=2):
            if len(fields) != len(header):
                issues.append(f"line {line_no}: expected {len(header)} fields, got {len(fields)}, row skipped")
                continue
            raw = dict(zip(header, fields))

            pid = raw["penguin_id"].strip()
            if pid in seen_ids:
                issues.append(f"line {line_no}: duplicate id {pid}, row skipped")
                continue
            seen_ids.add(pid)

            row = {
                "penguin_id": pid,
                "species": clean_category(raw["species"], SPECIES, suffix="penguin"),
                "island": raw["island"].strip().title(),
                "body_mass_g": parse_mass_grams(raw["body_mass"]),
                "sex": clean_category(raw["sex"], SEX),
                "date_seen": parse_date(raw["date_seen"]),
            }
            for col, raw_col in [("species", "species"), ("body_mass_g", "body_mass"),
                                 ("sex", "sex"), ("date_seen", "date_seen")]:
                if row[col] is None:
                    issues.append(f"line {line_no}: {pid} has no valid {col} (raw value: {raw[raw_col]!r})")
            clean.append(row)

    return clean, issues


clean_rows, issues = clean_field_notes(messy_path)

for row in clean_rows:
    print(row)
print()
for issue in issues:
    print(issue)

{'penguin_id': 'P001', 'species': 'Adelie', 'island': 'Torgersen', 'body_mass_g': 3750.0, 'sex': 'male', 'date_seen': '2007-11-11'}
{'penguin_id': 'P002', 'species': 'Adelie', 'island': 'Torgersen', 'body_mass_g': 3800.0, 'sex': 'female', 'date_seen': '2007-11-16'}
{'penguin_id': 'P003', 'species': 'Adelie', 'island': 'Torgersen', 'body_mass_g': None, 'sex': 'female', 'date_seen': '2007-11-16'}
{'penguin_id': 'P004', 'species': 'Gentoo', 'island': 'Biscoe', 'body_mass_g': 5200.0, 'sex': 'male', 'date_seen': '2008-11-13'}
{'penguin_id': 'P005', 'species': 'Gentoo', 'island': 'Biscoe', 'body_mass_g': None, 'sex': 'male', 'date_seen': '2008-11-13'}
{'penguin_id': 'P006', 'species': 'Chinstrap', 'island': 'Dream', 'body_mass_g': 3500.0, 'sex': None, 'date_seen': '2009-11-21'}
{'penguin_id': 'P008', 'species': 'Gentoo', 'island': 'Biscoe', 'body_mass_g': None, 'sex': 'female', 'date_seen': '2008-11-14'}
{'penguin_id': 'P009', 'species': 'Adelie', 'island': 'Torgersen', 'body_mass_g': 3650.0

Every row is now consistent, and the issue log says exactly what was dropped or left empty, and why. A few decisions are worth noting:

- **The broken row was skipped, not repaired.** We could guess that `3,500 g` meant 3500 grams, but guessing which comma is the stray one is fragile, and getting it wrong silently corrupts a row. With one broken row out of ten, skipping it and reporting it is safer. If it happened often, the right fix would be at the source: quote the values properly when the file is written.
- **Missing values stay as `None` for now.** Whether to drop those rows, or fill them in, depends on what the data is used for. That decision belongs in the modelling step, and it has its own rules (next section).
- **Duplicates are detected by ID**, with a set, so each check is O(1), as in Module 2.

### Filling missing values without leaking

If you fill in missing values, for example with the median of a column, there's a rule from Module 1 that applies again: **compute the fill value from the training data only**. If you compute the median from all the data, information about the test set leaks into training, and your test score becomes slightly optimistic. The mini project and the exercises follow this rule.

## 8. JSON and JSON Lines

CSV is good for flat tables. For anything with structure, like configs, nested results, API responses or model metadata, the standard format is **JSON**. Python's `json` module converts between JSON text and Python's dicts, lists, strings, numbers, booleans and `None`:

In [26]:
import json

config = {
    "model": "knn",
    "params": {"k": 5, "weights": "distance"},
    "features": ["bill_length_mm", "flipper_length_mm"],
    "scale": True,
    "notes": None,
}

text = json.dumps(config, indent=2)
print(text)

restored = json.loads(text)
print(restored == config)

{
  "model": "knn",
  "params": {
    "k": 5,
    "weights": "distance"
  },
  "features": [
    "bill_length_mm",
    "flipper_length_mm"
  ],
  "scale": true,
  "notes": null
}
True


`dumps`/`loads` (with an "s" for *string*) work with text. `dump`/`load` work with files directly:

In [27]:
with open(work / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

with open(work / "config.json", encoding="utf-8") as f:
    print(json.load(f)["params"])

{'k': 5, 'weights': 'distance'}


### What doesn't survive the round trip

JSON has fewer types than Python, so some things change on the way through. These three cause real bugs:

In [28]:
data = {
    "shape": (28, 28),              # tuple
    "label_names": {0: "Adelie", 1: "Gentoo"},   # int keys
}
back = json.loads(json.dumps(data))

print(back)
print(back["shape"] == (28, 28))         # it's a list now
print(back["label_names"].get(0))        # keys became strings
print(back["label_names"].get("0"))

{'shape': [28, 28], 'label_names': {'0': 'Adelie', '1': 'Gentoo'}}
False
None
Adelie


Tuples come back as lists, and **dictionary keys always come back as strings**, because JSON only allows string keys. Code that saves an `{id: label}` mapping and loads it back will find that `mapping[0]` raises a `KeyError`. Convert the keys back explicitly after loading: `{int(k): v for k, v in loaded.items()}`.

The third one is `NaN`:

In [29]:
print(json.dumps({"loss": float("nan")}))

try:
    json.dumps({"loss": float("nan")}, allow_nan=False)
except ValueError as e:
    print("ValueError:", e)

{"loss": NaN}
ValueError: Out of range float values are not JSON compliant: nan


Python writes `NaN` happily, but `NaN` isn't valid JSON, and many other tools (JavaScript, some databases, strict parsers) will refuse to read the file. Since a `nan` loss is exactly the kind of thing you'd want to log, convert it to `None` (which becomes JSON `null`) before saving, or use `allow_nan=False` so you find out early.

### Saving dataclasses

Results stored as dataclasses (Module 3) convert to JSON through `asdict`:

In [30]:
from dataclasses import dataclass, asdict


@dataclass
class RunResult:
    run_id: str
    accuracy: float
    params: dict


result = RunResult("run_007", 0.912, {"k": 5})
print(json.dumps(asdict(result)))

{"run_id": "run_007", "accuracy": 0.912, "params": {"k": 5}}


### JSON Lines

A JSON file is one big value, so you have to read all of it before you can use any of it. For datasets and logs, the usual format is **JSON Lines** (`.jsonl`): one JSON object per line. You can append to it one record at a time, and read it line by line with a generator, so memory stays flat. Many NLP datasets and most LLM fine-tuning data are distributed this way.

In [31]:
log_path = work / "training_log.jsonl"

with open(log_path, "w", encoding="utf-8") as f:
    for epoch, (loss, acc) in enumerate([(0.9, 0.61), (0.6, 0.74), (0.45, 0.81)]):
        f.write(json.dumps({"epoch": epoch, "loss": loss, "accuracy": acc}) + "\n")

print(log_path.read_text(encoding="utf-8"))


def read_jsonl(path: Path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)


best = max(read_jsonl(log_path), key=lambda r: r["accuracy"])
print("best epoch:", best)

{"epoch": 0, "loss": 0.9, "accuracy": 0.61}
{"epoch": 1, "loss": 0.6, "accuracy": 0.74}
{"epoch": 2, "loss": 0.45, "accuracy": 0.81}

best epoch: {'epoch': 2, 'loss': 0.45, 'accuracy': 0.81}


Another advantage: if a training run crashes, everything logged up to that point is still a valid file. A half-written JSON file usually can't be read at all.

## 9. A word about pickle

Python's `pickle` module can save almost any Python object to a file, including class instances, which JSON can't do. It's tempting to use it for everything. Two warnings:

**Never load a pickle file you don't trust.** Loading a pickle can run arbitrary code. A malicious pickle file can do anything your account can do, such as delete files or steal credentials. This is a real attack route, not a theoretical one: security researchers have found model files shared online that carried malicious code this way. It's why PyTorch's `torch.load` now defaults to `weights_only=True`, and why the `safetensors` format was created for sharing model weights.

**Pickles are fragile over time.** A pickle stores a reference to your class by name. Rename the class, move it to another module, or change its attributes, and old pickle files may fail to load or load incorrectly.

So: pickle is fine for short-lived caches of your own data on your own machine. For anything you share, or need to read again in six months, prefer a plain format like JSON or CSV, as the mini project does when it saves a trained model.

## 10. Putting it together: a reproducible data pipeline

Let's turn the raw penguin file into clean, model-ready train and test files. A good pipeline is **reproducible**: run it again and you get exactly the same files. That means fixed random seeds, no manual steps, and a record of every decision.

The steps:

1. Load the raw CSV and convert types.
2. Drop rows with no measurements.
3. Split into train and test, stratified by species, with a fixed seed.
4. Fill missing `sex` values with `"unknown"`. It's a category, so "unknown" is honest, and it doesn't use any statistics, so there's no leakage risk.
5. Save `train.csv`, `test.csv` and a `metadata.json` describing what was done.
6. Read everything back and check it.

In [32]:
import random
from collections import defaultdict


def stratified_split(rows: list[dict], key: str, test_ratio: float, seed: int):
    groups = defaultdict(list)
    for row in rows:
        groups[row[key]].append(row)
    rng = random.Random(seed)
    train, test = [], []
    for label in sorted(groups):              # sorted, so the order never depends on the data
        group = groups[label][:]
        rng.shuffle(group)
        n_test = round(len(group) * test_ratio)
        test += group[:n_test]
        train += group[n_test:]
    return train, test


def write_rows(rows: list[dict], path: Path, columns: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)

Iterating over `sorted(groups)` is a small but important detail. If we went in whatever order the species first appeared in the file, a re-ordered copy of the same data would shuffle differently. Sorting makes the split depend only on the data's content and the seed.

In [33]:
SEED = 42
COLUMNS = ["species", "island", "sex", "year"] + NUMERIC

raw_rows = load_penguins(get_penguins())
usable = [r for r in raw_rows if any(r[c] is not None for c in NUMERIC)]
train, test = stratified_split(usable, key="species", test_ratio=0.2, seed=SEED)

for row in train + test:
    if row["sex"] is None:
        row["sex"] = "unknown"

out = work / "processed"
write_rows(train, out / "train.csv", COLUMNS)
write_rows(test, out / "test.csv", COLUMNS)

metadata = {
    "source": "data/penguins.csv (Palmer Penguins, CC0)",
    "seed": SEED,
    "test_ratio": 0.2,
    "rows_raw": len(raw_rows),
    "rows_dropped_no_measurements": len(raw_rows) - len(usable),
    "rows_train": len(train),
    "rows_test": len(test),
    "species_train": dict(Counter(r["species"] for r in train)),
    "species_test": dict(Counter(r["species"] for r in test)),
    "sex_filled_with": "unknown",
    "columns": COLUMNS,
}
(out / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(json.dumps(metadata, indent=2))

{
  "source": "data/penguins.csv (Palmer Penguins, CC0)",
  "seed": 42,
  "test_ratio": 0.2,
  "rows_raw": 344,
  "rows_dropped_no_measurements": 2,
  "rows_train": 273,
  "rows_test": 69,
  "species_train": {
    "Adelie": 121,
    "Chinstrap": 54,
    "Gentoo": 98
  },
  "species_test": {
    "Adelie": 30,
    "Chinstrap": 14,
    "Gentoo": 25
  },
  "sex_filled_with": "unknown",
  "columns": [
    "species",
    "island",
    "sex",
    "year",
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g"
  ]
}


The metadata file is the pipeline's memory. Six months from now, it tells you exactly how `train.csv` was made: which seed, how many rows were dropped and why, what was filled in. Anyone can rerun the pipeline and check they get the same counts.

Finally, read the files back and check they're what we expect. A pipeline that writes files should always be tested by reading them, since bugs in writing (a wrong column order, a missing header) are easy to miss otherwise:

In [34]:
def read_processed(path: Path) -> list[dict]:
    with open(path, encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))
    for row in rows:
        row["year"] = int(row["year"])
        for col in NUMERIC:
            row[col] = float(row[col])
    return rows


train_back = read_processed(out / "train.csv")
test_back = read_processed(out / "test.csv")
saved_meta = json.loads((out / "metadata.json").read_text(encoding="utf-8"))

assert len(train_back) == saved_meta["rows_train"]
assert len(test_back) == saved_meta["rows_test"]
assert all(r[c] is not None for r in train_back + test_back for c in NUMERIC)
assert not ({tuple(r.values()) for r in train_back} & {tuple(r.values()) for r in test_back})

print("all checks passed")
print("first training row:", train_back[0])
print("files:", sorted(p.name for p in out.iterdir()))

all checks passed
first training row: {'species': 'Adelie', 'island': 'Dream', 'sex': 'female', 'year': 2008, 'bill_length_mm': 38.1, 'bill_depth_mm': 18.6, 'flipper_length_mm': 190.0, 'body_mass_g': 3700.0}
files: ['metadata.json', 'test.csv', 'train.csv']


The last `assert` checks that no row appears in both train and test. It turns each row into a tuple so it can go in a set (Module 2), and looks for any overlap. That's the data leakage check from Module 2, applied to real files.

`assert` is Python's built-in sanity check: it does nothing if the condition is true, and raises an `AssertionError` if it's false. It's great for checks like these in notebooks and scripts. Just don't rely on it for validating user input in production code, because Python can be run with assertions switched off.

## Summary

- Use `pathlib.Path` for paths: `/` to join, `.name`, `.suffix`, `.parent`, `.exists()`, `.mkdir(parents=True, exist_ok=True)`. Keep paths relative to the project.
- Open files with `with open(path, mode, encoding="utf-8")`. `"w"` wipes the file, `"a"` appends, `"x"` refuses to overwrite.
- Loop over a file object to read it lazily, line by line.
- Always set the encoding. Use `"utf-8-sig"` for CSVs that may come from Excel. Try `latin-1` or `cp1252` if UTF-8 fails.
- Download data once and cache it locally.
- Read CSV with the `csv` module and `newline=""`, never with `split(",")`. Every value is a string: convert types yourself and handle all the missing-value markers.
- Explore before modelling: row counts, class balance, missing values, per-class statistics.
- Clean with one small function per problem, map categories through an explicit dict, sanity-check numeric ranges, and log every issue instead of fixing things silently.
- Compute fill values from training data only.
- JSON for structured data: tuples become lists, keys become strings, and `NaN` isn't valid JSON. JSON Lines for logs and datasets.
- Never unpickle untrusted files. Prefer plain formats for anything shared or long-lived.
- Make pipelines reproducible: fixed seeds, sorted iteration, metadata files, and read-back checks.

Next, open `02_exercises.ipynb`.